# Training Happy and Pessimistic LLMs Using PPO (RLHF)

In this project, we explore how to train two language models—one that generates **positive, cheerful responses (“Happy LLM”)** and another that produces **negative, pessimistic responses (“Pessimistic LLM”)**—to simulate different behavioral styles in customer service applications.

To achieve this, we use a reward function derived from a sentiment classifier trained on the IMDb movie review dataset. This classifier evaluates the sentiment of generated text and provides feedback signals that guide the learning process.

---

## Reinforcement Learning Overview

Reinforcement Learning (RL) is a branch of machine learning where an agent learns by interacting with an environment and receiving feedback in the form of rewards or penalties. The goal of the agent is to learn a policy that maximizes cumulative reward over time.

In this context, the **language model acts as the agent**, and its actions correspond to generating text tokens. Unlike supervised learning, RL does not require labeled input-output pairs. Instead, the model improves through exploration and feedback from the reward signal.

---

## Proximal Policy Optimization (PPO)

Proximal Policy Optimization (PPO) is a widely used reinforcement learning algorithm introduced by OpenAI. It is known for its balance between performance and stability.

PPO directly optimizes the policy while ensuring that updates remain controlled and do not deviate too much from the previous policy. This constraint helps maintain stable training and avoids destructive updates during learning.

---

## Project Goal

In this lab, you will learn how to implement and train a reinforcement learning agent using the PPO algorithm, focusing on sentiment-based text generation.

You will work with the IMDb dataset, which contains a large collection of movie reviews used for sentiment analysis tasks. By the end of this exercise, you will understand how PPO can be applied to fine-tune language models and how reinforcement learning techniques can be extended to other natural language processing problems.

---

### Import Libraries

In [1]:
import torch
import torch.nn as nn
import pandas as pd
from datasets import load_dataset
import trl 
import json
from transformers import pipeline
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead
from trl.core import LengthSampler
from tqdm import tqdm
import matplotlib.pyplot as plt
from transformers import AutoTokenizer,AutoModelForCausalLM

d:\Projects\rlhf-ppo-sentiment-steering\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
trl.__version__

'0.8.6'

helper functions

In [3]:
def load_from_json(filepath):
    
    with open (filepath, "r") as file:
        content = json.load(file)
    
    return content

In [4]:
def save_json_to_file(filepath, data):
    
    with open(filepath, "w") as file:
        json.dump(data, file, indent=4)
    
    print(f"{filepath} write sucessfully")
    

### Load and Config Dataset

In [5]:
dataset = load_dataset("imdb")
dataset

Using the latest cached version of the dataset since imdb couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at C:\Users\Vish\.cache\huggingface\datasets\imdb\plain_text\0.0.0\e6281661ce1c48d982bc483cf8a173c1bbeb5d31 (last modified on Mon May 25 09:59:45 2026).


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [6]:
dataset_train = dataset['train']

In [7]:
for n in range(5):
    print("review: ",dataset_train['text'][n])
    print("label: ", dataset_train['label'][n])

review:  I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far bet

In [8]:
dataset_train = dataset_train.rename_column("text", "review")
dataset_train.column_names

['review', 'label']

Filter the reviews that's length is greater than 200

In [9]:
dataset_filter = dataset_train.filter(
    lambda data: len(data['review'])>200, batched=False
)

In [10]:
dataset_filter[0:10]

{'review': ['I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far

Using a ```LengthSampler``` to sample different text lengths during data processing introduces variability, making the model more robust and capable of handling varying input lengths in real-world scenarios. This approach prevents overfitting by exposing the model to diverse input sizes, improving generalization to new data. It also ensures efficient training by managing the length of text inputs, maintaining practicality and performance. Overall, LengthSampler enhances model adaptability and effectiveness by simulating realistic, varied training conditions. Where sample length is between ```input_min_text_length``` and ```input_max_text_length```








In [11]:
input_min_text_len, input_max_text_len = 2,8

In [12]:
input_size = LengthSampler(input_min_text_len, input_max_text_len)

### PPO Configuration

In [13]:
ppo_config = PPOConfig(
    model_name="lvwerra/gpt2-imdb",
    learning_rate=1.41e-5
)

`config.model_name` is the parameter that defines which pretrained model will be loaded in the configuration. It points to a model on the Hugging Face model hub. Here, it is set to `"lvwerra/gpt2-imdb"`, meaning a GPT-2 model that has already been fine-tuned on the IMDB dataset by the user *lvwerra* will be used. This setting is important because it ensures the correct model structure and pretrained weights are retrieved for either training or inference.


In [14]:
ppo_config.model_name

'lvwerra/gpt2-imdb'

In [15]:
sent_kwargs = {"top_k": None, "function_to_apply": None, "batch_size": 2}

In [16]:
model = AutoModelForCausalLMWithValueHead.from_pretrained(ppo_config.model_name,)

Loading weights: 100%|██████████| 149/149 [00:00<00:00, 16560.27it/s]


During PPO training, update the model. In addition, the reference model is used to stabilize the model using the Kullback-Leibler (KL) divergence between the current policy and the reference policy.The KL divergence acts as a regularization term.

**So we need to get a reference model to comparing process**

In [17]:
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(ppo_config.model_name)

Loading weights: 100%|██████████| 149/149 [00:00<00:00, 13537.93it/s]


In [18]:
tokenizer = AutoTokenizer.from_pretrained(ppo_config.model_name)
tokenizer.pad_token = tokenizer.eos_token

In [19]:
def text_tokenized (dataset):
    dataset['input_ids'] = tokenizer.encode(dataset['review'])[:input_size()]
    dataset['query'] = tokenizer.decode(dataset['input_ids'])
    
    return dataset

In [20]:
dataset_filter = dataset_filter.map(text_tokenized,batched=False)
dataset_filter

Dataset({
    features: ['review', 'label', 'input_ids', 'query'],
    num_rows: 24895
})

In [21]:
dataset_filter[0]

{'review': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far 

In [22]:
dataset_filter.set_format(type="torch")

In [23]:
for i, sample in enumerate(dataset_filter):
    if i >= 5:
        break
    print(f"Sample {i+1}:")
    print(f"Review: {sample['review']}")
    print(f"Input IDs: {sample['input_ids']}")
    print(f"Query: {sample['query']}")
    print("-" * 50)

Sample 1:
Review: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few an

#### Collator Function

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device.type

'cpu'

In [25]:
def collator(dataset):
    return dict((k,[data[k] for data in dataset]) for k in dataset[0])

In [26]:
batch = collator(dataset_filter)

In [27]:
##batch

### PPOTrainer Initialization and Overview

Proximal Policy Optimization (PPO) is a reinforcement learning algorithm widely used for training generative models, especially in applications like chatbots. It helps solve key challenges in training language models, such as maintaining stable learning and producing coherent, context-aware responses.

PPO improves traditional policy gradient methods by introducing a clipped objective function. This ensures that updates to the model remain small and controlled, preventing unstable or drastic changes during training. As a result, the chatbot maintains consistent response quality across training iterations.

Unlike standard policy gradient approaches, which often suffer from high variance and unstable behavior, PPO provides a more reliable training process. It achieves this by balancing exploration of new responses with exploitation of already learned good responses, using a trust-region-like constraint.

The PPO training process involves collecting dialogue samples from the model, evaluating them using a reward signal, and then updating the model’s policy based on these rewards. This cycle is managed by the PPOTrainer, which handles model optimization, sample collection, and training coordination.

By using PPOTrainer, the training process becomes more stable and efficient, ultimately leading to chatbot responses that are more natural, coherent, and contextually appropriate.

In [28]:
ppo__trainer = PPOTrainer(
    config=ppo_config,
    model = model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    data_collator=collator,
    dataset=dataset_filter,
)
print(f"PPO trainer: {ppo__trainer}")

PPO trainer: <trl.trainer.ppo_trainer.PPOTrainer object at 0x0000022A8DC77310>


In [29]:
## device config
device = ppo__trainer.accelerator.device
device

device(type='cpu')

## Reward Function

In reinforcement learning using PPO (Proximal Policy Optimization), the reward function plays a key role in measuring how good or bad an action is. For generative AI systems such as chatbots, the reward function is used to assess the quality of the generated responses and provide feedback that helps improve future outputs.

One simple approach is to use a sentiment analysis pipeline as the reward mechanism. The generated response is analyzed to determine whether its sentiment is positive, negative, or neutral. Based on the sentiment score, a numerical reward is assigned to the chatbot’s response.

During PPO training, these rewards guide the model to adjust its policy so that it produces responses that are more engaging, meaningful, and positively perceived by users. The PPO Trainer continuously learns from this feedback and updates the chatbot’s behavior over time.

Although sentiment analysis is not considered a traditional reward model, it offers an easy and practical method for introducing reinforcement learning concepts into chatbot training. This technique demonstrates how external evaluation systems can be used to improve conversational AI performance effectively.

In [30]:
reward_model_pipeline = pipeline(task="sentiment-analysis", model="lvwerra/distilbert-imdb", device = device)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 17331.15it/s]


In [31]:
sample = "This is a fun movie i ever watched"
reward_model_pipeline(sample, **sent_kwargs)

[{'label': 'POSITIVE', 'score': 0.9948694705963135},
 {'label': 'NEGATIVE', 'score': 0.005130556412041187}]

## Generating responses using PPO 

In [32]:
batch = next(iter(ppo__trainer.dataloader))
batch.keys()

dict_keys(['label', 'input_ids', 'query'])

In [33]:
batch = {k:batch[k][0:2] for k in batch.keys()}
batch   

{'label': [tensor(0), tensor(0)],
 'input_ids': [tensor([19197,   645,  3241,   284,   262]),
  tensor([1212, 3807,  373, 7818,   13])],
 'query': ['Pay no attention to the', 'This movie was terrible.']}

In [113]:
response_tensors = [] #initiate a list to store the response tensors from the model


In [114]:
query_tensor = batch['input_ids']
query_tensor

[tensor([19197,   645,  3241,   284,   262]),
 tensor([1212, 3807,  373, 7818,   13])]

**Define function to Decode the Responses**

In [115]:
def get_text (tensor_list):
    decode_list = " ".join([tokenizer.decode(response.squeeze()) for response in tensor_list])
    return decode_list

In [116]:
get_text(query_tensor)

'Pay no attention to the This movie was terrible.'

#### PPO Generate

In [117]:
query_1 = query_tensor[0]
query_1

tensor([19197,   645,  3241,   284,   262])

In [118]:
generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": 50256
}

In [119]:
output_min_length = 4
output_max_length = 20

output_length = LengthSampler(output_min_length, output_max_length) 


In [120]:
gen_len = output_length()

In [121]:
generation_kwargs["max_new_tokens"] = gen_len

In [122]:
response_1 = ppo__trainer.generate(query_1,**generation_kwargs)

In [123]:
response_1

tensor([[19197,   645,  3241,   284,   262,  3307,   286,   262, 26049,   338,
          6027,  1218,  3800,   711,    11,   287]])

In [124]:
print(f"Query: {get_text(query_1)}")
print(f"response: {get_text(response_1)}")

Query: Pay  no  attention  to  the
response: Pay no attention to the details of the Opera's planned second stage play, in


Append the each response tensor values to response tensors list: 

In [135]:
def query_to_response_tensor(query_tensor):
    response_tensor = []
    for query in query_tensor:
        output = ppo__trainer.generate(query, **generation_kwargs)
        response_tensor.append(output.squeeze()[-gen_len:])
    return response_tensor

In [125]:
response_tensors.append(response_1.squeeze()[-gen_len:])
print("newly generated tokens form response:", get_text(response_tensors[0]))

newly generated tokens form response:  details  of  the  Opera 's  planned  second  stage  play ,  in


In [126]:
def pad_seq_to_len(tensor, pad_token_id, length):
    
    padding_length = length - tensor.size(0)
    if padding_length > 0:
        padding = torch.full(padding_length,pad_token_id,dtype=torch.long, device=tensor.device)
        return torch.cat((tensor,padding))
    
    return tensor

In [147]:
def pad_list_to_batch_size(tensors, batch_size, pad_token_id):
    max_len_tensor = max(t.size(0) for t in tensors)
    padded_tensors = [pad_seq_to_len(t,pad_token_id,max_len_tensor) for t in tensors]
    
    if len(padded_tensors) < batch_size:
        padded_tensors.append(torch.full((max_len_tensor,), pad_token_id,dtype=torch.long, device=tensors[0].device))
    
    return padded_tensors[:batch_size]

To meet the PPO trainer's minimum batch size requirement of 128, you can pad the response tensors with additional sample.


In [148]:
response_tensor = query_to_response_tensor(query_tensor)

In [149]:
response_tensor

[tensor([  636,   286,   262,  3807,   326, 30742,   262,  3872,   546,   509,
          5549]),
 tensor([  314,  1254,   264, 13321,    78,  6507,   326,   314,  7342,   340,
            13])]

In [150]:
batch_size = 128
pad_token_id = tokenizer.pad_token_id

query_tensor = pad_list_to_batch_size(query_tensor, batch_size, pad_token_id)
response_tensors = pad_list_to_batch_size(response_tensor, batch_size, pad_token_id)


In [151]:
query_tensor, response_tensors

([tensor([19197,   645,  3241,   284,   262]),
  tensor([1212, 3807,  373, 7818,   13]),
  tensor([50256, 50256, 50256, 50256, 50256])],
 [tensor([  636,   286,   262,  3807,   326, 30742,   262,  3872,   546,   509,
           5549]),
  tensor([  314,  1254,   264, 13321,    78,  6507,   326,   314,  7342,   340,
             13]),
  tensor([50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
          50256])])